In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import efficientnet
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras import mixed_precision

In [ ]:

# Path to the images
# IMAGES_PATH = "Flicker8k_Dataset"

# Desired image dimensions
IMAGE_SIZE = (299, 299)

# Vocabulary size
VOCAB_SIZE = 10000

# Fixed length allowed for any sequence
SEQ_LENGTH = 25

# Dimension for the image embeddings and token embeddings
EMBED_DIM = 512

# Per-layer units in the feed-forward network
FF_DIM = 512

# Model Architecture Vairables
NUM_HEADS = 8
NUM_LAYERS = 3

# Other training parameters
IMG_SIZE = 224
BATCH_SIZE = 8
SEED = 10
EPOCHS = 120
AUTO = tf.data.AUTOTUNE

In [ ]:
train_data = pd.read_csv("/kaggle/input/datasets/tomtillo/satellite-image-caption-generation/train.csv")
valid_data = pd.read_csv("/kaggle/input/datasets/tomtillo/satellite-image-caption-generation/valid.csv")
test_data = pd.read_csv("/kaggle/input/datasets/tomtillo/satellite-image-caption-generation/test.csv")

In [ ]:
# Use this if running natively in Kaggle Notebooks:
BASE_PATH = "/kaggle/input/datasets/tomtillo/satellite-image-caption-generation"
# If running in Colab (after downloading/unzipping the dataset), use your extraction path like:
# BASE_PATH = "/content/satellite-image-caption-generation"

def process_satellite_csv(csv_filename):
    """Loads satellite dataset safely without aggressively dropping short/long captions."""
    df = pd.read_csv(os.path.join(BASE_PATH, csv_filename))
    
    caption_mapping = {}
    text_data =[]
    
    for _, row in df.iterrows():
        img_path = os.path.join(BASE_PATH, row['filepath'])
        raw_caps_str = str(row['captions'])
        
        # 1. Clean the string syntax
        clean_str = raw_caps_str.replace('[', '').replace(']', '')
        
        # 2. Split by newline (which separates the 5 captions in this specific dataset)
        if '\n' in clean_str:
            caps_list = clean_str.split('\n')
        else:
            caps_list = clean_str.split(',')
            
        valid_caps_for_img =[]
        
        for caption in caps_list:
            # 3. Clean up residual quotes and spaces
            caption = caption.strip().strip("'").strip('"').strip()
            
            if not caption: 
                continue
                
            # NOTE: We completely removed the `len(tokens) < 5` filter.
            # Keras TextVectorization automatically handles padding and truncating!
            formatted_caption = "<start> " + caption + " <end>"
            valid_caps_for_img.append(formatted_caption)
            
        if valid_caps_for_img:
            caption_mapping[img_path] = valid_caps_for_img

    if not caption_mapping:
        return {},[]

    # 4. Enforce perfectly rectangular shapes (5 captions per image) for TensorFlow TypeSpec
    target_len = max(len(caps) for caps in caption_mapping.values())
    
    final_mapping = {}
    for img_path, caps in caption_mapping.items():
        # If an image has fewer than target_len captions (e.g. due to a blank line in CSV), duplicate the last one
        if len(caps) < target_len:
            caps = caps + [caps[-1]] * (target_len - len(caps))
            
        # If somehow there are more, truncate to target_len
        caps = caps[:target_len]
            
        final_mapping[img_path] = caps
        text_data.extend(caps)
            
    return final_mapping, text_data

In [ ]:
# Load Train and Validation splits directly from the CSV files
train_data, train_text_data = process_satellite_csv("train.csv")
valid_data, valid_text_data = process_satellite_csv("valid.csv")

# Combine all text data to build the vocabulary
text_data = train_text_data + valid_text_data

print("Number of training samples: ", len(train_data))
print("Number of validation samples: ", len(valid_data))

In [ ]:
def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(lowercase, "[%s]" % re.escape(strip_chars), "")


strip_chars = "!\"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"
strip_chars = strip_chars.replace("<", "")
strip_chars = strip_chars.replace(">", "")

vectorization = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=SEQ_LENGTH,
    standardize=custom_standardization,
)
vectorization.adapt(text_data)

# Data augmentation for image data
image_augmentation = keras.Sequential(
    [
        # layers.RandomFlip("horizontal"),
        # layers.RandomRotation(0.2),
        layers.RandomContrast(0.3),
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(factor=1.0), # Full 360 degree rotation
        # layers.RandomZoom(height_factor=(-0.2, 0.2)),
        # layers.RandomBrightness(factor=0.3),
    ]
)

In [ ]:
def decode_and_resize(img_path, size=IMAGE_SIZE):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    return img


def read_train_image(img_path, size=IMAGE_SIZE):
    img = decode_and_resize(img_path)
    img = image_augmentation(tf.expand_dims(img, 0))[0]
    img = tf.image.convert_image_dtype(img, tf.float32)
    return img


def read_valid_image(img_path, size=IMAGE_SIZE):
    img = decode_and_resize(img_path)
    img = tf.image.convert_image_dtype(img, tf.float32)
    return img


def make_dataset(images, captions, split="train"):
    if split == "train":
        img_dataset = tf.data.Dataset.from_tensor_slices(images).map(
            read_train_image, num_parallel_calls=AUTO
        )
    else:
        img_dataset = tf.data.Dataset.from_tensor_slices(images).map(
            read_valid_image, num_parallel_calls=AUTO
        )

    cap_dataset = tf.data.Dataset.from_tensor_slices(captions).map(
        vectorization, num_parallel_calls=AUTO
    )

    dataset = tf.data.Dataset.zip((img_dataset, cap_dataset))
    dataset = dataset.batch(BATCH_SIZE).shuffle(256).prefetch(AUTO)
    return dataset


# Pass the list of images and the list of corresponding captions
train_dataset = make_dataset(
    list(train_data.keys()), list(train_data.values()), split="train"
)

valid_dataset = make_dataset(
    list(valid_data.keys()), list(valid_data.values()), split="valid"
)

In [ ]:
# !pip install transformers -q

# from transformers import TFViTModel

# # ViT-B/16 requires exactly 224×224 — update this at the top of your notebook
# IMAGE_SIZE = (224, 224)  # was (299, 299)


# class ViTBackbone(layers.Layer):
#     """Wraps HuggingFace TFViTModel as a Keras-compatible layer."""
#     def __init__(self, model_name="google/vit-base-patch16-224-in21k", **kwargs):
#         super().__init__(**kwargs)
#         self.vit = TFViTModel.from_pretrained(model_name)
#         self.vit.trainable = False  # Freeze initially

#     def call(self, inputs, training=False):
#         # Normalize [0, 1] → [-1, 1]  (ViT ImageNet convention)
#         x = (inputs - 0.5) / 0.5

#         # TF is BHWC, HuggingFace ViT expects BCHW
#         x = tf.transpose(x, perm=[0, 3, 1, 2])

#         outputs = self.vit(pixel_values=x, training=training)

#         # last_hidden_state: (B, 197, 768)
#         # Token 0 is the CLS token — drop it, keep the 196 patch tokens
#         return outputs.last_hidden_state[:, 1:, :]  # → (B, 196, 768)


# def get_vit_model():
#     inputs = layers.Input(shape=(*IMAGE_SIZE, 3))

#     # Patch embeddings: (B, 196, 768)
#     x = ViTBackbone(name="vit_b16")(inputs)

#     # Project 768 → EMBED_DIM to match the rest of your transformer
#     output = layers.Dense(EMBED_DIM, activation="relu")(x)  # → (B, 196, EMBED_DIM)

#     return keras.Model(inputs, output, name="vit_backbone")

# # Building the model  
# vit_model = get_vit_model()   # replaces: cnn_model = get_cnn_model()

In [ ]:
# ==========================================
# 2. CNN Feature Extractor
# ==========================================

def get_cnn_model():
    base_model = efficientnet.EfficientNetB7(
        input_shape=(*IMAGE_SIZE, 3),
        include_top=False,
        weights="imagenet",
    )
    base_model.trainable = False

    base_model_out = base_model.output
    base_model_out = layers.BatchNormalization()(base_model_out)
    base_model_out = layers.Reshape((-1, base_model_out.shape[-1]))(base_model_out)

    # ✅ Fix 1: Project 2048 → EMBED_DIM so encoder residuals work
    base_model_out = layers.Dense(EMBED_DIM, activation="silu")(base_model_out)

    cnn_model = keras.models.Model(base_model.input, base_model_out)
    return cnn_model


# ==========================================
# 3. Transformer Components
# ==========================================

class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings    = layers.Embedding(input_dim=vocab_size,        output_dim=embed_dim)
        self.position_embeddings = layers.Embedding(input_dim=sequence_length,   output_dim=embed_dim)
        self.sequence_length     = sequence_length
        self.vocab_size          = vocab_size
        self.embed_dim           = embed_dim

    def call(self, inputs):
        length             = tf.shape(inputs)[-1]
        positions          = tf.range(start=0, limit=length, delta=1)
        embedded_tokens    = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return tf.math.not_equal(inputs, 0)


class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.attention  = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn        = keras.Sequential([
            layers.Dense(dense_dim, activation="silu"),
            layers.Dense(embed_dim),   # ✅ outputs embed_dim to match residual
        ])
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()

    def call(self, inputs, training):
        attn_output = self.attention(inputs, inputs, training=training)
        out1        = self.layernorm1(inputs + attn_output)
        ffn_output  = self.ffn(out1)
        return self.layernorm2(out1 + ffn_output)


class TransformerDecoderBlock(layers.Layer):
    def __init__(self, embed_dim, ff_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.attention_1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.attention_2 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn         = keras.Sequential([
            layers.Dense(ff_dim,    activation="silu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1  = layers.LayerNormalization()
        self.layernorm2  = layers.LayerNormalization()
        self.layernorm3  = layers.LayerNormalization()

    def call(self, inputs, encoder_outputs, training, mask=None):
        causal_mask = self.get_causal_attention_mask(inputs)
        if mask is not None:
            padding_mask  = tf.cast(mask[:, tf.newaxis, :], dtype="int32")
            combined_mask = tf.minimum(padding_mask, causal_mask)
        else:
            combined_mask = causal_mask

        attn_1 = self.attention_1(query=inputs, value=inputs, key=inputs,
                                  attention_mask=combined_mask, training=training)
        out_1  = self.layernorm1(inputs + attn_1)

        attn_2 = self.attention_2(query=out_1, value=encoder_outputs,
                                  key=encoder_outputs, training=training)
        out_2  = self.layernorm2(out_1 + attn_2)

        ffn_out = self.ffn(out_2)
        return self.layernorm3(out_2 + ffn_out)

    def get_causal_attention_mask(self, inputs):
        input_shape     = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i    = tf.range(sequence_length)[:, tf.newaxis]
        j    = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        return tf.reshape(mask, (1, sequence_length, sequence_length))


# ==========================================
# 4. Encoder and Decoder Wrappers
# ==========================================

class CaptionEncoder(layers.Layer):
    def __init__(self, num_layers, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.enc_blocks = [
            TransformerEncoderBlock(embed_dim, dense_dim, num_heads)
            for _ in range(num_layers)
        ]

    def call(self, x, training=False):
        for block in self.enc_blocks:
            x = block(x, training=training)
        return x


class CaptionDecoder(layers.Layer):
    def __init__(self, num_layers, embed_dim, ff_dim, num_heads, vocab_size, **kwargs):
        super().__init__(**kwargs)
        self.embedding  = PositionalEmbedding(SEQ_LENGTH, vocab_size, embed_dim)
        self.dropout    = layers.Dropout(0.5)
        self.dec_blocks = [
            TransformerDecoderBlock(embed_dim, ff_dim, num_heads)
            for _ in range(num_layers)
        ]
        self.out = layers.Dense(vocab_size, activation="softmax")

    def call(self, x, encoder_outputs, training=False, mask=None):
        x = self.embedding(x)
        x = self.dropout(x, training=training)
        for block in self.dec_blocks:
            x = block(x, encoder_outputs, training=training, mask=mask)
        return self.out(x)


# ==========================================
# 5. Full Model Class
# ==========================================

# ✅ Fix 2: Label smoothing loss (replaces SparseCategoricalCrossentropy)
def smooth_loss(y_true, y_pred, smoothing=0.1):
    loss        = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    smooth_term = -tf.reduce_mean(tf.math.log(y_pred + 1e-9), axis=-1)
    return (1.0 - smoothing) * loss + smoothing * smooth_term


class ImageCaptioningModel(keras.Model):
    def __init__(self, cnn_model, encoder, decoder, num_captions_per_image=5):
        super().__init__()
        self.cnn_model              = cnn_model
        self.encoder                = encoder
        self.decoder                = decoder
        self.num_captions_per_image = num_captions_per_image
        self.loss_tracker           = keras.metrics.Mean(name="loss")
        self.acc_tracker            = keras.metrics.Mean(name="acc")

    def calculate_loss(self, y_true, y_pred, mask):
        # ✅ Fix 3: use smooth_loss which returns per-token values
        loss = smooth_loss(y_true, y_pred)
        mask = tf.cast(mask, dtype=loss.dtype)
        loss *= mask
        return tf.reduce_sum(loss) / tf.reduce_sum(mask)

    def calculate_accuracy(self, y_true, y_pred, mask):
        accuracy = tf.equal(y_true, tf.argmax(y_pred, axis=2))
        accuracy = tf.math.logical_and(mask, accuracy)
        accuracy = tf.cast(accuracy, dtype=tf.float32)
        mask     = tf.cast(mask,     dtype=tf.float32)
        return tf.reduce_sum(accuracy) / tf.reduce_sum(mask)

    def train_step(self, batch_data):
        batch_img, batch_seq = batch_data
        img_features         = self.cnn_model(batch_img)
        img_features         = tf.repeat(img_features,
                                         repeats=self.num_captions_per_image, axis=0)
        batch_seq_flat       = tf.reshape(batch_seq, (-1, tf.shape(batch_seq)[2]))

        with tf.GradientTape() as tape:
            encoder_out = self.encoder(img_features, training=True)
            inputs      = batch_seq_flat[:, :-1]
            targets     = batch_seq_flat[:, 1:]
            mask        = tf.math.not_equal(targets, 0)
            preds       = self.decoder(inputs, encoder_out, training=True, mask=mask)
            loss        = self.calculate_loss(targets, preds, mask)
            acc         = self.calculate_accuracy(targets, preds, mask)

        trainable_vars = (self.encoder.trainable_variables +
                          self.decoder.trainable_variables)
        grads = tape.gradient(loss, trainable_vars)
        self.optimizer.apply_gradients(zip(grads, trainable_vars))

        self.loss_tracker.update_state(loss)
        self.acc_tracker.update_state(acc)
        return {"loss": self.loss_tracker.result(), "acc": self.acc_tracker.result()}

    def test_step(self, batch_data):
        batch_img, batch_seq = batch_data
        img_features         = self.cnn_model(batch_img)
        img_features         = tf.repeat(img_features,
                                         repeats=self.num_captions_per_image, axis=0)
        batch_seq_flat       = tf.reshape(batch_seq, (-1, tf.shape(batch_seq)[2]))

        encoder_out = self.encoder(img_features, training=False)
        inputs      = batch_seq_flat[:, :-1]
        targets     = batch_seq_flat[:, 1:]
        mask        = tf.math.not_equal(targets, 0)
        preds       = self.decoder(inputs, encoder_out, training=False, mask=mask)
        loss        = self.calculate_loss(targets, preds, mask)
        acc         = self.calculate_accuracy(targets, preds, mask)

        self.loss_tracker.update_state(loss)
        self.acc_tracker.update_state(acc)
        return {"loss": self.loss_tracker.result(), "acc": self.acc_tracker.result()}

    @property
    def metrics(self):
        return [self.loss_tracker, self.acc_tracker]


# ==========================================
# 6. Implementation & Training Setup
# ==========================================

cnn_model     = get_cnn_model()
encoder       = CaptionEncoder(NUM_LAYERS, EMBED_DIM, FF_DIM, NUM_HEADS)
decoder       = CaptionDecoder(NUM_LAYERS, EMBED_DIM, FF_DIM, NUM_HEADS, VOCAB_SIZE)

caption_model = ImageCaptioningModel(cnn_model, encoder, decoder)

caption_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0)
    # ✅ No loss= here — smooth_loss is called inside calculate_loss directly
)

In [ ]:
# Define the loss function
cross_entropy = keras.losses.SparseCategoricalCrossentropy(
    from_logits=False, reduction="none"
)

# EarlyStopping criteria
early_stopping = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)


# Learning Rate Scheduler for the optimizer
class LRSchedule(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, post_warmup_learning_rate, warmup_steps):
        super().__init__()
        self.post_warmup_learning_rate = post_warmup_learning_rate
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        global_step = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(self.warmup_steps, tf.float32)
        warmup_progress = global_step / warmup_steps
        warmup_learning_rate = self.post_warmup_learning_rate * warmup_progress
        return tf.cond(
            global_step < warmup_steps,
            lambda: warmup_learning_rate,
            lambda: self.post_warmup_learning_rate,
        )


# Create a learning rate schedule
num_train_steps = len(train_dataset) * EPOCHS
num_warmup_steps = num_train_steps // 15
lr_schedule = LRSchedule(post_warmup_learning_rate=1e-4, warmup_steps=num_warmup_steps)

# Compile the model
caption_model.compile(optimizer=keras.optimizers.Adam(lr_schedule), loss=cross_entropy)

# Fit the model
caption_model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=valid_dataset,
    callbacks=[early_stopping],
)

In [ ]:
vocab = vectorization.get_vocabulary()
index_lookup = dict(zip(range(len(vocab)), vocab))
max_decoded_sentence_length = SEQ_LENGTH - 1
valid_images = list(valid_data.keys())


def generate_caption():
    # Select a random image from the validation dataset
    sample_img = np.random.choice(valid_images)

    # Read the image from the disk
    sample_img = read_valid_image(sample_img)
    img = sample_img.numpy().clip(0, 255).astype(np.uint8)
    plt.imshow(img)
    plt.show()

    # Pass the image to the CNN
    img = tf.expand_dims(sample_img, 0)
    img = caption_model.cnn_model(img)

    # Pass the image features to the Transformer encoder
    encoded_img = caption_model.encoder(img, training=False)

    # Generate the caption using the Transformer decoder
    decoded_caption = "<start> "
    for i in range(max_decoded_sentence_length):
        tokenized_caption = vectorization([decoded_caption])[:, :-1]
        mask = tf.math.not_equal(tokenized_caption, 0)
        predictions = caption_model.decoder(
            tokenized_caption, encoded_img, training=False, mask=mask
        )
        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = index_lookup[sampled_token_index]
        if sampled_token == " <end>":
            break
        decoded_caption += " " + sampled_token

    decoded_caption = decoded_caption.replace("<start> ", "")
    decoded_caption = decoded_caption.replace(" <end>", "").strip()
    print("Predicted Caption: ", decoded_caption)


# Check predictions for a few samples
generate_caption()

In [ ]:
# import tensorflow as tf
# import numpy as np

# 1. SETUP: Create mappings to work with Token IDs directly
# We need to find the ID for "<start>" and "<end>" from your vectorization layer
word_to_index = tf.keras.layers.StringLookup(
    mask_token="",
    vocabulary=vectorization.get_vocabulary(),
    invert=False
)
index_to_word = tf.keras.layers.StringLookup(
    mask_token="",
    vocabulary=vectorization.get_vocabulary(),
    invert=True
)

start_token_id = word_to_index(tf.constant("<start>")).numpy()
end_token_id = word_to_index(tf.constant("<end>")).numpy()

def generate_caption_beam_search(k=3):
    # Select a random image
    sample_img_path = np.random.choice(valid_images)
    sample_img = read_valid_image(sample_img_path)

    # Display image (optional)
    img_display = sample_img.numpy().clip(0, 255).astype(np.uint8)
    plt.imshow(img_display)
    plt.show()

    # Pass the image to the CNN
    img = tf.expand_dims(sample_img, 0)
    img = caption_model.cnn_model(img)

    # Encode the image
    encoded_img = caption_model.encoder(img, training=False)

    # --- BEAM SEARCH LOGIC ---

    # Beam shape: [cumulative_log_prob, list_of_token_ids]
    # We start with 1 beam containing only the <start> token and 0 score
    beams = [(0.0, [start_token_id])]

    for i in range(max_decoded_sentence_length):
        all_candidates = []

        # Expand each current beam
        for score, seq in beams:
            # If this beam has already finished, don't expand it, just add it to candidates
            if seq[-1] == end_token_id:
                all_candidates.append((score, seq))
                continue

            # Prepare input for decoder
            # We must recreate the tensor shape (1, seq_len)
            tokenized_caption = tf.constant([seq], dtype=tf.int64)

            # Create mask (standard transformer masking)
            mask = tf.math.not_equal(tokenized_caption, 0)

            # Run Decoder
            predictions = caption_model.decoder(
                tokenized_caption, encoded_img, training=False, mask=mask
            )

            # Get logits for the LAST token in the sequence
            # Shape: (batch, seq_len, vocab_size) -> (vocab_size)
            last_token_logits = predictions[0, -1, :]

            # Convert logits to log-probabilities
            log_probs = tf.nn.log_softmax(last_token_logits)

            # Get top k (beam width) next words to save computation
            top_k_log_probs, top_k_indices = tf.math.top_k(log_probs, k=k)

            # Add these new candidates to the pile
            for j in range(k):
                new_score = score + top_k_log_probs[j].numpy()
                new_seq = seq + [top_k_indices[j].numpy()]
                all_candidates.append((new_score, new_seq))

        # Sort all candidates by score (highest log-prob first)
        ordered = sorted(all_candidates, key=lambda x: x[0], reverse=True)

        # Keep only the top k beams
        beams = ordered[:k]

        # Stop early if the best beam has finished
        if beams[0][1][-1] == end_token_id:
            break

    # --- DECODING THE BEST SEQUENCE ---

    # Take the best beam (index 0)
    best_seq = beams[0][1]

    # Convert IDs back to words
    decoded_caption = " ".join([index_lookup[idx] for idx in best_seq])

    # Clean up tokens
    decoded_caption = decoded_caption.replace("<start>", "").replace("<end>", "").strip()
    print("Predicted Caption (Beam Search): ", decoded_caption)

# Run it
for i in range (10):
    generate_caption_beam_search(k=5)


In [ ]:
from IPython.display import display as ipy_display

In [ ]:
def generate_custom_image_caption(img_path, k=3, max_length=SEQ_LENGTH):
    # 1. Load and preprocess the custom image
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)

    # Display the image — using ipy_display to force render inside function
    img_display = img.numpy().clip(0, 255).astype(np.uint8)
    fig, ax = plt.subplots()
    ax.imshow(img_display)
    ax.set_title("Custom Input Image")
    ax.axis("off")
    ipy_display(fig)       # <-- this is the key fix
    plt.close(fig)         # prevents duplicate renders

    # Convert to float32 [0,1] for the model
    img_tensor = tf.image.convert_image_dtype(img, tf.float32)

    # 2. Pass the image to the CNN & Encoder
    img_batch = tf.expand_dims(img_tensor, 0)
    cnn_features = caption_model.cnn_model(img_batch)
    encoded_img = caption_model.encoder(cnn_features, training=False)

    # 3. BEAM SEARCH LOGIC
    beams = [(0.0, [start_token_id])]
    for i in range(max_length):
        all_candidates = []
        for score, seq in beams:
            if seq[-1] == end_token_id:
                all_candidates.append((score, seq))
                continue
            tokenized_caption = tf.constant([seq], dtype=tf.int64)
            mask = tf.math.not_equal(tokenized_caption, 0)
            predictions = caption_model.decoder(
                tokenized_caption, encoded_img, training=False, mask=mask
            )
            last_token_logits = predictions[0, -1, :]
            log_probs = tf.nn.log_softmax(last_token_logits)
            top_k_log_probs, top_k_indices = tf.math.top_k(log_probs, k=k)
            for j in range(k):
                new_score = score + top_k_log_probs[j].numpy()
                new_seq = seq + [top_k_indices[j].numpy()]
                all_candidates.append((new_score, new_seq))

        ordered = sorted(all_candidates, key=lambda x: x[0], reverse=True)
        beams = ordered[:k]
        if beams[0][1][-1] == end_token_id:
            break

    # 4. DECODING THE BEST SEQUENCE
    best_seq = beams[0][1]
    decoded_words = [vocab[idx] for idx in best_seq if idx < len(vocab)]
    decoded_caption = " ".join(decoded_words)
    decoded_caption = decoded_caption.replace("<start>", "").replace("<end>", "").replace("[UNK]", "").strip()

    return decoded_caption

In [ ]:
import random
import matplotlib.pyplot as plt

test_image_paths = random.sample(list(test_data_processed.keys()), 20)

for img_path in test_image_paths:
    try:
        caption = generate_caption_for_image(img_path, k=1)
    except Exception as e:
        caption = f"[ERROR: {e}]"

    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = img.numpy()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f'Caption: "{caption}"\n')

In [ ]:
import random
import matplotlib.pyplot as plt

test_image_paths = random.sample(list(test_data_processed.keys()), 15)

for img_path in test_image_paths:
    try:
        caption = generate_caption_for_image(img_path, k=2)
    except Exception as e:
        caption = f"[ERROR: {e}]"

    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = img.numpy()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f'Caption: "{caption}"\n')

In [ ]:
import os
import matplotlib.pyplot as plt

US_IRAN_FOLDER = "/kaggle/input/datasets/rogaldorn7/urban-landscapes"
VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".tif", ".tiff")

image_paths = [
    os.path.join(US_IRAN_FOLDER, f)
    for f in sorted(os.listdir(US_IRAN_FOLDER))
    if f.lower().endswith(VALID_EXTENSIONS)
]

for img_path in image_paths:
    try:
        caption = generate_caption_for_image(img_path, k=1)
    except Exception as e:
        caption = f"[ERROR: {e}]"

    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = img.numpy()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f'Caption: "{caption}"\n')

In [ ]:
import os
import matplotlib.pyplot as plt

US_IRAN_FOLDER = "/kaggle/input/datasets/rogaldorn7/urban-landscapes"
VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".tif", ".tiff")

image_paths = [
    os.path.join(US_IRAN_FOLDER, f)
    for f in sorted(os.listdir(US_IRAN_FOLDER))
    if f.lower().endswith(VALID_EXTENSIONS)
]

for img_path in image_paths:
    try:
        caption = generate_caption_for_image(img_path, k=2)
    except Exception as e:
        caption = f"[ERROR: {e}]"

    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = img.numpy()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f'Caption: "{caption}"\n')

In [ ]:
import os
import matplotlib.pyplot as plt

US_IRAN_FOLDER = "/kaggle/input/datasets/rogaldorn7/us-iran-war"
VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".tif", ".tiff")

image_paths = [
    os.path.join(US_IRAN_FOLDER, f)
    for f in sorted(os.listdir(US_IRAN_FOLDER))
    if f.lower().endswith(VALID_EXTENSIONS)
]

for img_path in image_paths:
    try:
        caption = generate_caption_for_image(img_path, k=1)
    except Exception as e:
        caption = f"[ERROR: {e}]"

    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = img.numpy()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f'Caption: "{caption}"\n')

In [ ]:
import os
import matplotlib.pyplot as plt

US_IRAN_FOLDER = "/kaggle/input/datasets/rogaldorn7/us-iran-war"
VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".tif", ".tiff")

image_paths = [
    os.path.join(US_IRAN_FOLDER, f)
    for f in sorted(os.listdir(US_IRAN_FOLDER))
    if f.lower().endswith(VALID_EXTENSIONS)
]

for img_path in image_paths:
    try:
        caption = generate_caption_for_image(img_path, k=2)
    except Exception as e:
        caption = f"[ERROR: {e}]"

    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = img.numpy()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f'Caption: "{caption}"\n')

Model Performance Evaluation

In [ ]:
# Step 1: Process the test split (same pipeline as train/valid)
test_data_processed, test_text_data = process_satellite_csv("test.csv")

test_dataset = make_dataset(
    list(test_data_processed.keys()),
    list(test_data_processed.values()),
    split="valid"   # use "valid" mode — no augmentation, no shuffling
)

# Step 2: Evaluate
print("Evaluating on test set...")
results = caption_model.evaluate(test_dataset, verbose=1)

# Step 3: Print cleanly
print("\n" + "="*35)
print(f"  Test Loss     : {results[0]:.4f}")
print(f"  Test Accuracy : {results[1]*100:.2f}%")
print("="*35)

BLEU Score

In [ ]:
# Install if not already available
!pip install nltk -q
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

In [ ]:
!pip install tqdm -q
from tqdm import tqdm

In [ ]:
# ============================================================
# Step 1: Single-image caption generator (reusable base)
# ============================================================

def generate_caption_for_image(img_path, k=5):
    """
    Runs beam search on a specific image path and returns
    the predicted caption as a plain string.
    """
    # Load and preprocess
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.image.convert_image_dtype(img, tf.float32)

    # Extract features
    img_batch   = tf.expand_dims(img, 0)
    cnn_out     = caption_model.cnn_model(img_batch, training=False)
    encoded_img = caption_model.encoder(cnn_out, training=False)

    # Beam search
    beams = [(0.0, [start_token_id])]

    for _ in range(max_decoded_sentence_length):
        all_candidates = []

        for score, seq in beams:
            if seq[-1] == end_token_id:
                all_candidates.append((score, seq))
                continue

            tokenized = tf.constant([seq], dtype=tf.int64)
            mask      = tf.math.not_equal(tokenized, 0)
            preds     = caption_model.decoder(
                tokenized, encoded_img, training=False, mask=mask
            )

            log_probs            = tf.nn.log_softmax(preds[0, -1, :])
            top_k_probs, top_k_ids = tf.math.top_k(log_probs, k=k)

            for j in range(k):
                new_score = score + top_k_probs[j].numpy()
                new_seq   = seq + [top_k_ids[j].numpy()]
                all_candidates.append((new_score, new_seq))

        # Length-normalised sort (prevents bias toward short captions)
        beams = sorted(
            all_candidates,
            key=lambda x: x[0] / (len(x[1]) ** 0.7),
            reverse=True
        )[:k]

        if beams[0][1][-1] == end_token_id:
            break

    # Decode best beam to string
    best_ids = beams[0][1]
    words    = [
        vocab[idx] for idx in best_ids
        if idx < len(vocab)
           and vocab[idx] not in ("<start>", "<end>", "[UNK]", "")
    ]
    return " ".join(words).strip()

In [ ]:
def evaluate_bleu4(data_dict, k=3, max_samples=500):
    smoother   = SmoothingFunction().method1
    hypotheses = []
    references = []

    img_paths = list(data_dict.keys())[:max_samples]

    for img_path in tqdm(img_paths, desc="Evaluating BLEU", unit="img"):
        try:
            pred = generate_caption_for_image(img_path, k=k)
        except Exception:
            continue

        pred_tokens = nltk.word_tokenize(pred.lower())

        raw_caps = data_dict[img_path]
        ref_token_lists = [
            nltk.word_tokenize(
                cap.replace("<start>", "").replace("<end>", "").strip().lower()
            )
            for cap in raw_caps
        ]

        hypotheses.append(pred_tokens)
        references.append(ref_token_lists)

    bleu1 = corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0),             smoothing_function=smoother)
    bleu2 = corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0, 0),         smoothing_function=smoother)
    bleu3 = corpus_bleu(references, hypotheses, weights=(0.33, 0.33, 0.33, 0),    smoothing_function=smoother)
    bleu4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoother)

    print("\n" + "=" * 40)
    print(f"  Samples evaluated : {len(hypotheses)}")
    print(f"  BLEU-1            : {bleu1*100:.2f}%")
    print(f"  BLEU-2            : {bleu2*100:.2f}%")
    print(f"  BLEU-3            : {bleu3*100:.2f}%")
    print(f"  BLEU-4            : {bleu4*100:.2f}%")
    print("=" * 40)

    return {
        "bleu1": round(bleu1 * 100, 2),
        "bleu2": round(bleu2 * 100, 2),
        "bleu3": round(bleu3 * 100, 2),
        "bleu4": round(bleu4 * 100, 2),
    }

In [ ]:
bleu_scores = evaluate_bleu4(
    data_dict   = test_data_processed,
    k           = 5,       # beam width — match what you used during training
    max_samples = 800,    # set e.g. 100 for a quick sanity check first
)

In [ ]:
word_to_index = tf.keras.layers.StringLookup(
    mask_token="",
    vocabulary=vectorization.get_vocabulary(),
    invert=False
)
index_to_word = tf.keras.layers.StringLookup(
    mask_token="",
    vocabulary=vectorization.get_vocabulary(),
    invert=True
)

vocab                    = vectorization.get_vocabulary()
start_token_id           = word_to_index(tf.constant("<start>")).numpy()
end_token_id             = word_to_index(tf.constant("<end>")).numpy()
max_decoded_sentence_length = SEQ_LENGTH - 1

In [ ]:
bleu_scores = evaluate_bleu4(
    data_dict   = test_data_processed,
    k           = 4,       # beam width — match what you used during training
    max_samples = 800,    # set e.g. 100 for a quick sanity check first
)

In [ ]:
bleu_scores = evaluate_bleu4(
    data_dict   = test_data_processed,
    k           = 3,       # beam width — match what you used during training
    max_samples = 800,    # set e.g. 100 for a quick sanity check first
)

In [ ]:
bleu_scores = evaluate_bleu4(
    data_dict   = test_data_processed,
    k           = 2,       # beam width — match what you used during training
    max_samples = 800,    # set e.g. 100 for a quick sanity check first
)

In [ ]:
bleu_scores = evaluate_bleu4(
    data_dict   = test_data_processed,
    k           = 1,       # beam width — match what you used during training
    max_samples = 800,    # set e.g. 100 for a quick sanity check first
)